Merge matches files with points files with match_id.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1]
DATA_RAW = ROOT / "data_raw"

POINTS_RAW  = DATA_RAW / "points_raw"
MATCHES_RAW = DATA_RAW / "matches_raw"

assert POINTS_RAW.exists()
assert MATCHES_RAW.exists()

print("Project root:", ROOT)

Project root: /Users/leventezsiga/Documents/VU/Thesis_P2-P3/tennis-pressure


Load all matches files into a single DataFrame.

In [ ]:
matches_files = sorted(MATCHES_RAW.glob("*.csv"))
assert len(matches_files) > 0, "No match files found"

print(f"Found {len(matches_files)} matches files")

matches_dfs = []
for f in matches_files:
    df = pd.read_csv(f)
    df["source_file"] = f.name
    matches_dfs.append(df)

matches_df = pd.concat(matches_dfs, ignore_index=True)

print("Matches dataframe shape:", matches_df.shape)
print("Unique matches:", matches_df["match_id"].nunique())


Found 49 matches files
Matches dataframe shape: (10513, 17)
Unique matches: 10513


In [6]:
assert "match_id" in matches_df.columns

dup_matches = matches_df["match_id"].duplicated().sum()
print("Duplicate match_id rows:", dup_matches)


Duplicate match_id rows: 0


In [ ]:
points_files = sorted(POINTS_RAW.glob("*.csv"))
assert len(points_files) > 0, "No points files found"

print(f"Found {len(points_files)} points files")

points_dfs = []
for f in points_files:
    df = pd.read_csv(f)
    df["source_file"] = f.name
    points_dfs.append(df)

points_df = pd.concat(points_dfs, ignore_index=True)

print("Points dataframe shape:", points_df.shape)
print("Unique match_id (points):", points_df["match_id"].nunique())

Found 49 points files


/var/folders/sk/00rkx0hd027cbgk6d5rxrl7w0000gn/T/ipykernel_16379/4243251827.py:10: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


Points dataframe shape: (1922136, 66)
Unique match_id (points): 10508


The following columns will be kept in the merged dataset.

In [9]:
keep_matches = ["match_id", "player1", "player2", "round", "year", "slam"]
keep_points = [
    "match_id", "ElapsedTime", "SetNo", "GameNo", "PointNumber",
    "PointServer", "PointWinner",
    "P1Score", "P2Score",
    "P1GamesWon", "P2GamesWon", "P1PointsWon", "P2PointsWon", "Rally",
    "P1BreakPoint", "P2BreakPoint", "P1BreakPointWon", "P2BreakPointWon"
]

Matches files merged with points on 'match_id'.

In [10]:
matches_sub = matches_df[keep_matches].copy()
points_sub = points_df[keep_points].copy()

In [11]:
print("Matches subset shape:", matches_sub.shape)
print("Points subset shape:", points_sub.shape)

Matches subset shape: (10513, 6)
Points subset shape: (1922136, 18)


In [ ]:
merged_df = points_sub.merge(
    matches_sub,
    on="match_id",
    how="left",
    validate="many_to_one"
)

In [ ]:
print("Merged dataframe shape:", merged_df.shape)
print("Unique match_id (merged):", merged_df["match_id"].nunique())

Merged dataframe shape: (1922136, 23)
Unique match_id (merged): 10508


In [ ]:
bad_match_ids = (
    merged_df.loc[merged_df["player1"].isna() | merged_df["player2"].isna(), "match_id"]
    .dropna()
    .unique()
)
print("Bad match_ids (either player missing):", len(bad_match_ids))
print("Example:", bad_match_ids[:10])

Bad match_ids (either player missing): 32
Example: ['2016-usopen-2148' '2016-usopen-2163' '2016-usopen-2224'
 '2016-usopen-2303' '2016-usopen-2402' '2017-ausopen-2109'
 '2017-ausopen-2116' '2017-ausopen-2132' '2017-ausopen-2145'
 '2017-ausopen-2205']


In [ ]:
merged_clean = merged_df[~merged_df["match_id"].isin(bad_match_ids)].copy()

print("Clean merged shape:", merged_clean.shape)
print("Clean unique match_id:", merged_clean["match_id"].nunique())

Clean merged shape: (1917826, 23)
Clean unique match_id: 10476


In [ ]:
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

pd.Series(bad_match_ids, name="match_id").to_csv(
    OUTPUTS / "dropped_matches_missing_metadata.csv",
    index=False
)
print("Logged dropped match_ids to outputs/")

Logged dropped match_ids to outputs/


In [ ]:
assert merged_clean["player1"].isna().sum() == 0
assert merged_clean["player2"].isna().sum() == 0

print(
    "Dropped matches:",
    merged_df["match_id"].nunique() - merged_clean["match_id"].nunique()
)

Dropped matches: 32


Merged clean should now contain only matches with all match and point-by-point data.

Upon inspecting the .csv files, it is visible that the WTA data, for each tournament is recorded consecutively after the ATP. The research focuses only on WTA matches, and only on rounds of the quarterfinals, semifinals and finals. Therefore filtering for the last 7 unique 'match_id'-s for each 'year' and 'slam', will leave the desired set.

In [ ]:
merged_df = merged_df.sort_values(["year", "slam", "match_id"])

In [32]:
keep_ids = (
    merged_df
    .groupby(["year", "slam"])["match_id"]
    .unique()
    .apply(lambda x: sorted(x)[-7:])
    .explode()
    .tolist()
)

In [34]:
qf_df = merged_df[merged_df["match_id"].isin(keep_ids)].reset_index(drop=True)

Sanity checks.

In [ ]:
print("Unique matches kept:", qf_df["match_id"].nunique())

Unique matches kept: 343


Print the first name appearing as 'player1' in the dataset for each slam in each year.

In [40]:
qf_df.groupby(["year", "slam"])["player1"].first()

year  slam      
2011  ausopen        Caroline Wozniacki
      frenchopen    Svetlana Kuznetsova
      usopen             Sabine Lisicki
      wimbledon      Dominika Cibulkova
2012  ausopen              Ana Ivanovic
      frenchopen       Klara Zakopalova
      usopen          Victoria Azarenka
      wimbledon          Sabine Lisicki
2013  ausopen         Victoria Azarenka
      frenchopen        Serena Williams
      usopen            Serena Williams
      wimbledon          Sabine Lisicki
2014  ausopen              Ana Ivanovic
      frenchopen       Garbine Muguruza
      usopen            Serena Williams
      wimbledon        Eugenie Bouchard
2015  ausopen           Serena Williams
      frenchopen        Serena Williams
      usopen            Serena Williams
      wimbledon         Serena Williams
2016  ausopen           Serena Williams
      frenchopen        Serena Williams
      usopen            Serena Williams
      wimbledon             Sam Querrey
2017  ausopen          

The 2016 Wimbledon is the odd one out, most likely WTA data was not recorded for that tournament. This tournament will be dropped.

In [41]:
qf_df = qf_df[
    ~((qf_df["year"] == 2016) & (qf_df["slam"] == "wimbledon"))
].reset_index(drop=True)


In [42]:
qf_df.groupby(["year", "slam"])["player1"].first()

year  slam      
2011  ausopen        Caroline Wozniacki
      frenchopen    Svetlana Kuznetsova
      usopen             Sabine Lisicki
      wimbledon      Dominika Cibulkova
2012  ausopen              Ana Ivanovic
      frenchopen       Klara Zakopalova
      usopen          Victoria Azarenka
      wimbledon          Sabine Lisicki
2013  ausopen         Victoria Azarenka
      frenchopen        Serena Williams
      usopen            Serena Williams
      wimbledon          Sabine Lisicki
2014  ausopen              Ana Ivanovic
      frenchopen       Garbine Muguruza
      usopen            Serena Williams
      wimbledon        Eugenie Bouchard
2015  ausopen           Serena Williams
      frenchopen        Serena Williams
      usopen            Serena Williams
      wimbledon         Serena Williams
2016  ausopen           Serena Williams
      frenchopen        Serena Williams
      usopen            Serena Williams
2017  ausopen           CoCo Vandeweghe
      frenchopen       

Now the remaining dataset contains all the available WTA matches with point-by-point data from the quarterfinals and later rounds (From now on referenced as 'QF+ matches'). The completeness and correctness of the point-by-point data will be determined later, some matches may be dropped later on.

Necessarry typecasting before savings.

In [51]:
int_cols = [
    "SetNo", "GameNo", "PointNumber",
    "PointWinner", "PointServer",
    "P1GamesWon", "P2GamesWon",
    "P1PointsWon", "P2PointsWon",
    "P1BreakPoint", "P2BreakPoint",
    "P1BreakPointWon", "P2BreakPointWon",
    "P1Score", "P2Score"
]

for col in int_cols:
    if col in qf_df.columns:
        qf_df[col] = (
            pd.to_numeric(qf_df[col], errors="coerce")
              .astype("Int64")
        )

if "ElapsedTime" in qf_df.columns:
    qf_df["ElapsedTime"] = pd.to_timedelta(
        qf_df["ElapsedTime"], errors="coerce"
    )

qf_df = (
    qf_df
    .sort_values(["match_id", "SetNo", "GameNo", "PointNumber"])
    .reset_index(drop=True)
)

Save.

In [49]:
MERGED_DIR = DATA_RAW / "matches_points_merged"
print("Saving to:", MERGED_DIR)

Saving to: /Users/leventezsiga/Documents/VU/Thesis_P2-P3/tennis-pressure/data_raw/matches_points_merged


In [52]:
out_path = MERGED_DIR / "merged_WTA_QF+_raw.parquet"

qf_df.to_parquet(out_path, index=False)

print("Saved parquet:", out_path)
print("Shape:", qf_df.shape)
print("Unique match_id:", qf_df["match_id"].nunique())

Saved parquet: /Users/leventezsiga/Documents/VU/Thesis_P2-P3/tennis-pressure/data_raw/matches_points_merged/merged_WTA_QF+_raw.parquet
Shape: (48211, 23)
Unique match_id: 336
